<a href="https://colab.research.google.com/github/sngillard/student-outcome-classification/blob/main/notebooks/03_logistic_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load processed dataset from GitHub URL
df = pd.read_csv('https://raw.githubusercontent.com/sngillard/student-outcome-classification/refs/heads/main/data/processed/student_data_processed_2026-01-08.csv')

# Split dataset into features and target
X = df.drop(columns=['Target'])
y = df['Target']

# Split data into training and testing sets with 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a Logistic Regression model from sklearn  (B2- not trained or tuned yet)
logistic_regression_model = LogisticRegression(solver='lbfgs', max_iter=5000, random_state=42
                                    )
print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)


Training shape: (3539, 34)
Testing shape: (885, 34)


In [3]:
# Train the model using the AI/ML algorithm.
logistic_regression_model.fit(X_train, y_train)

# Test the model runs/makes classification predictions
y_prediction_log = logistic_regression_model.predict(X_test)
print('Testing predictions:', y_prediction_log[:20])


Testing predictions: [2 2 0 2 2 2 2 2 2 2 2 2 0 0 1 1 0 0 0 0]


The above predictions make sense based on the dataset distribution for the target variable that can be viewed in the bar chart found in notebook 02_exploratory_data_analysis.ipynb

In [ ]:
# Evaluate model accuracy using these metrics: accuracy, precision, recall, f1-score, and confusion matrix.
y_prediction_log = logistic_regression_model.predict(X_test)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,  confusion_matrix, classification_report

accuracy_score_log = accuracy_score(y_test, y_prediction_log)
precision_score_log = precision_score(y_test, y_prediction_log, average='weighted')
recall_score_log = recall_score(y_test, y_prediction_log, average='weighted')
f1_score_log = f1_score(y_test, y_prediction_log, average='weighted')
confusion_matrix_log = confusion_matrix(y_test, y_prediction_log)

print('Logistic Multinomial Regression Model- Evaluation Metrics:')
print(f'Accuracy: {accuracy_score_log:.4f}')
print(f'Precision (weighted): {precision_score_log:.4f}')
print(f'Recall (weighted): {recall_score_log:.4f}')
print(f'F1 Score (weighted): {f1_score_log:.4f}')
print('Confusion Matrix for Logistic Regression Model:')
print(confusion_matrix_log)
print(classification_report(y_test, y_prediction_log))


Logistic Multinomial Regression Model- Evaluation Metrics:
Accuracy: 0.7571
Precision (weighted): 0.7389
Recall (weighted): 0.7571
F1 Score (weighted): 0.7421
Confusion Matrix for Logistic Regression Model:
[[213  30  41]
 [ 41  52  66]
 [ 16  21 405]]
              precision    recall  f1-score   support

           0       0.79      0.75      0.77       284
           1       0.50      0.33      0.40       159
           2       0.79      0.92      0.85       442

    accuracy                           0.76       885
   macro avg       0.69      0.66      0.67       885
weighted avg       0.74      0.76      0.74       885



### Results from metrics for Multinomial Logistic Regression Classifier Model:
* **Accuracy**: 0.7551
* **Precision**: 0.7389
* **Recall**:0.7571
* **F1-Score:** 0.7421
* **Confusion Matrix**: <br>
 213        30          41 <br>
 41         52          66 <br>
16          21          405
*  **Classification matrix**: *see above*

### Model Evaluation summary:
* Before cross-validation and hyperparameter tuning, the models have similar performance based on the metrics above.
* The gradient boosting model has *slightly* better accuracy than the multinomial logistic regression model.

In [ ]:
# Apply cross-validation techniques.
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# Create stratified k-fold cross-validation object
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation with accuracy score
cv_scores_logr = cross_val_score(logistic_regression_model, X, y, cv=skf, scoring='accuracy')

print('Logistic regression cross-validation scores:', cv_scores_logr)
print('Logistic regression mean cross-validation accuracy:', np.mean(cv_scores_logr))
print('Logistic regression cross-validation standard deviation:', np.std(cv_scores_logr))

Logistic regression cross-validation scores: [0.7740113  0.7740113  0.76949153 0.75706215 0.74660633]
Logistic regression mean cross-validation accuracy: 0.7642365212056139
Logistic regression cross-validation standard deviation: 0.010779636044061494


### Comparing Models After Cross-Validation
* Gradient boosting **cross-validation scores**: [0.77542373 0.78389831 0.78248588 0.76836158 0.7708628 ]
* Logistic regression **cross-validation scores**: [0.7740113  0.7740113  0.76949153 0.75706215 0.74660633]
* Gradient boosting **mean cross-validation accuracy**: 0.7762064584182389
* Logistic regression **mean cross-validation accuracy**: 0.7642365212056139
* Gradient boosting **cross-validation standard deviation**: 0.006153129589961597
* Logistic regression **cross-validation standard deviation**: 0.010779636044061494

Applying stratified 5-fold cross-validation helped to analyze the performance of each model.
   
The **gradient boosting model** produced a slightly **higher mean cross-validation accuracy** with **lower variance** shown in the std score compared to the multinomial logistic regression model.

These findings indicate that the **gradient boosting model has a *slightly* more consistent performance across folds**.

In [ ]:
# Use hyperparameter tuning to optimize the model.
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Create a Logistic Regression model from sklearn  (B2- not trained or tuned yet)
logistic_regression_model = LogisticRegression(solver='lbfgs', max_iter=5000, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid_logr = {
    'C': [0.01, 0.1, 1.0, 5.0, 10.0, 100.0],
    'solver': ['lbfgs'],
    'penalty': ['l2'],
}

grid_logr = GridSearchCV(
    estimator=logistic_regression_model,
    param_grid=param_grid_logr,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_logr.fit(X_train, y_train)

print('Best logistic regression params:', grid_logr.best_params_)
print('Best logistic regression cross-validation accuracy:', grid_logr.best_score_)

best_logr_model = grid_logr.best_estimator_

y_prediction_logr_tuned = best_logr_model.predict(X_test)

print('Tuned logistic regression model- Test Accuracy:', accuracy_score(y_test, y_prediction_logr_tuned))
print('Tuned logistic regression model- Precision (weighted):', precision_score(y_test, y_prediction_logr_tuned, average='weighted'))
print('Tuned logistic regression model- Recall (weighted)', recall_score(y_test, y_prediction_logr_tuned, average='weighted'))
print('Tuned logistic regression model- F1-Score (weighted):', f1_score(y_test, y_prediction_logr_tuned, average='weighted'))
print('Tuned logistic regression model- Confusion Matrix:\n', confusion_matrix(y_test, y_prediction_logr_tuned))
print('Tuned logistic regression model- Classification Report:\n', classification_report(y_test, y_prediction_logr_tuned))


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best logistic regression params: {'C': 10.0, 'penalty': 'l2', 'solver': 'lbfgs'}
Best logistic regression cross-validation accuracy: 0.7711153197644219
Tuned logistic regression model- Test Accuracy: 0.7548022598870057
Tuned logistic regression model- Precision (weighted): 0.7358316481166226
Tuned logistic regression model- Recall (weighted) 0.7548022598870057
Tuned logistic regression model- F1-Score (weighted): 0.7395969934440639
Tuned logistic regression model- Confusion Matrix:
 [[212  31  41]
 [ 43  51  65]
 [ 16  21 405]]
Tuned logistic regression model- Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.75      0.76       284
           1       0.50      0.32      0.39       159
           2       0.79      0.92      0.85       442

    accuracy                           0.75       885
   macro avg       0.69      0.66      0.67       885
weighted avg       0.74

## Hyperparameter Tuning- Multinomial Logistic Regression Model
- GridsearchCV was used to optimize the model and tune the hyperparameters. Weighted metrics helped address class imbalance.

### Best Tuned hyperparameters- Gradient boosting model:
- C = 1.0
- Penalty = L2
- Solver = lbfgs

### Best cross-validation accuracy: 0.77

### Model performance for tuned model:
- Accuracy: 0.75
- Precision (weighted): 0.74
- Recall (weighted): 0.75
- F1-score (weighted): 0.74